# 02 · Cleaning — WDBC Breast Cancer (Tier A)

**Stage 2 of [STRUCTURE.md](../DOCS/STRUCTURE.md).** The data notes report **no real missing values and no
duplicate `id`s**, so this stage is mostly *validation and structural decisions* — we **verify** those
claims independently rather than assume them, encode the target, and record every action in a cleaning log.

**Three structural decisions live here:** (1) drop `id`, (2) encode the target `M=1/B=0`, (3) retain the
zero-inflated concavity features and the large-nucleus tails as **biological signal, not error**. Feature
engineering (ratios, PCA) is deliberately *deferred* to the modelling Pipeline in notebook 04 to avoid
leakage.

In [1]:
# --- DESIGN.md palette + matplotlib style (inlined; notebooks import no local modules) ---
import matplotlib as mpl
import matplotlib.pyplot as plt

NAVY   = "#051C2C"   # ink only, never a series fill
BLUE   = "#2251FF"   # McKinsey blue -> emphasis / Malignant (at-risk class)
TEAL   = "#00857C"   # secondary series -> Benign
CYAN   = "#00A9F4"   # tertiary categorical
AMBER  = "#C1841C"   # reference / decision-threshold lines
SLATE  = "#7F93A6"   # muted labels
GREY   = "#9FADB8"   # neutral context (non-highlighted)
GRID   = "#E9ECEF"   # gridlines
CAT    = [BLUE, TEAL, AMBER, CYAN, SLATE]           # fixed categorical order
DIAG   = {"M": BLUE, "B": TEAL}                      # semantic: Malignant=Blue, Benign=Teal

def apply_style():
    mpl.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica Neue", "DejaVu Sans"],
        "font.size": 10.5, "axes.titlesize": 12.5, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 10,
        "axes.edgecolor": SLATE, "axes.linewidth": 0.8, "axes.labelcolor": NAVY,
        "axes.spines.top": False, "axes.spines.right": False,
        "text.color": NAVY, "xtick.color": NAVY, "ytick.color": NAVY,
        "grid.color": GRID, "grid.linewidth": 0.8, "axes.grid": True, "axes.grid.axis": "y",
        "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
    })

apply_style()

from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "data.csv"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(RAW)
df = raw.drop(columns=[c for c in raw.columns if raw[c].isna().all()])  # drop phantom col again (idempotent)
print("loaded:", df.shape)
log = []  # cleaning log: one row per action
def logit(action, cols, rows, why):
    log.append({"action": action, "columns": cols, "rows_affected": rows, "why": why})

loaded: (569, 32)


## 2.1 Verify the notes — don't assume

Re-derive missingness and duplicate counts from scratch. If they contradict the data notes, the notes lose.

In [2]:
n_null = int(df.isna().sum().sum())
n_dup_id = int(df["id"].duplicated().sum())
n_dup_rows = int(df.drop(columns="id").duplicated().sum())
print(f"total nulls (post phantom-drop): {n_null}")
print(f"duplicate ids: {n_dup_id}")
print(f"fully-duplicate feature rows: {n_dup_rows}")
assert n_null == 0 and n_dup_id == 0, "notes contradicted — investigate before proceeding"
logit("verify missingness", "all", 0, f"independently confirmed {n_null} nulls after phantom-col drop")
logit("verify duplicates", "id + full-row", 0, f"{n_dup_id} dup ids, {n_dup_rows} dup rows — one row per sample")

total nulls (post phantom-drop): 0
duplicate ids: 0
fully-duplicate feature rows: 0


## 2.2 Drop `id`, encode the target

`id` is an opaque key with no analytic value (data notes §3) — it was only needed for the duplicate check
above. The target is encoded **M=1 (positive, by cost) / B=0**.

In [3]:
df = df.drop(columns="id")
logit("drop id", "id", 569, "opaque key, no analytic value; duplicate check already done")

df["diagnosis"] = pd.Categorical(df["diagnosis"], categories=["B", "M"], ordered=True)
df["diagnosis_binary"] = (df["diagnosis"] == "M").astype("int8")
logit("encode target", "diagnosis -> diagnosis_binary", 569, "M=1 (positive class by cost), B=0")
print(df["diagnosis"].value_counts())
print("positive rate (M):", round(df["diagnosis_binary"].mean(), 4))

diagnosis
B    357
M    212
Name: count, dtype: int64
positive rate (M): 0.3726


## 2.3 Types

All 30 measurements are continuous → `float64`. `diagnosis` is an ordered category; `diagnosis_binary`
is `int8`. No column is a disguised index or code (contrast the Boston `RAD` case) — so no categorical
recoding is needed.

In [4]:
FEATURES = [c for c in df.columns if c not in ("diagnosis", "diagnosis_binary")]
df[FEATURES] = df[FEATURES].astype("float64")
logit("cast features", "30 morphology features", 569, "continuous measurements -> float64")
assert len(FEATURES) == 30
print(df.dtypes.value_counts())

float64     30
category     1
int8         1
Name: count, dtype: int64


## 2.4 Outliers & zero-inflation — documented, retained

Two properties look like defects but are **genuine biology**, so both are retained (STRUCTURE.md: *never
remove outliers without a documented reason*):

- **Large-nucleus tails** (`area_worst`, `concavity_worst`, …): malignant masses genuinely produce larger,
  more irregular nuclei. Deleting them would delete the malignancy signal. Handled later by standardization,
  never by removal.
- **Zero-inflation**: 13 samples have exactly `0` concavity/concave-points across all three summaries —
  benign masses with no concave contour. This favours non-linear/tree models and matters for the modelling
  choice; recorded as a flag, not scrubbed.

In [5]:
conc_cols = [f"{b}_{s}" for b in ["concavity","concave points"] for s in ["mean","se","worst"]]
zero_rows = (df[conc_cols] == 0).any(axis=1)
n_zero = int(zero_rows.sum())
by_class = df.loc[zero_rows, "diagnosis"].value_counts().to_dict()
print(f"samples with any zero concavity/concave-points: {n_zero}  by class: {by_class}")

df["zero_concavity_flag"] = zero_rows.astype("int8")
logit("flag zero-inflation", "concavity/concave-points -> zero_concavity_flag", n_zero,
      f"{n_zero} benign-dominated samples with no concave contour; modelling signal, retained")

# quantify the large-nucleus tails (report, do not remove)
tail = df[FEATURES].apply(lambda s: (s > s.mean() + 3*s.std()).sum())
print("features with >0 points beyond mean+3sd:", int((tail > 0).sum()), "/ 30 (retained as biology)")
logit("retain right-tail outliers", "area/concavity/… worst", int(tail.sum()),
      "genuine malignant morphology; standardized at model time, never deleted")

samples with any zero concavity/concave-points: 13  by class: {'B': 13, 'M': 0}
features with >0 points beyond mean+3sd: 29 / 30 (retained as biology)


## 2.5 Business-rule validation on the cleaned frame

In [6]:
assert (df[FEATURES] >= 0).all().all(), "negative measurement found"
assert df["diagnosis_binary"].isin([0,1]).all()
assert df.isna().sum().sum() == 0
assert df.shape[0] == 569
print("business rules pass · final cleaned shape:", df.shape)

business rules pass · final cleaned shape: (569, 33)


## 2.6 Persist processed data + cleaning log

In [7]:
out = df[["diagnosis", "diagnosis_binary", "zero_concavity_flag"] + FEATURES]
out.to_parquet(PROC / "wdbc_clean.parquet", index=False)
clean_log = pd.DataFrame(log)
clean_log.to_csv(PROC / "_cleaning_log.csv", index=False)
print("saved:", (PROC / "wdbc_clean.parquet").relative_to(ROOT).as_posix(), "->", out.shape)
clean_log

saved: data/processed/wdbc_clean.parquet -> (569, 33)


,action,columns,rows_affected,why
0,verify missingness,all,0,independently confirmed 0 nulls after phantom-...
1,verify duplicates,id + full-row,0,"0 dup ids, 0 dup rows — one row per sample"
2,drop id,id,569,"opaque key, no analytic value; duplicate check..."
3,encode target,diagnosis -> diagnosis_binary,569,"M=1 (positive class by cost), B=0"
4,cast features,30 morphology features,569,continuous measurements -> float64
5,flag zero-inflation,concavity/concave-points -> zero_concavity_flag,13,13 benign-dominated samples with no concave co...
6,retain right-tail outliers,area/concavity/… worst,210,genuine malignant morphology; standardized at ...


## Stage 2 — Gate Checklist

- [x] **Missing-value strategy documented per column** — independently verified **0 nulls** (only the phantom column existed, dropped in Stage 1); nothing imputed
- [x] **Duplicates assessed & handled with justification** — 0 duplicate `id`s, 0 duplicate rows → one row per FNA sample; `id` then dropped
- [x] **All columns have correct dtypes** — 30 features `float64`, `diagnosis` ordered category, `diagnosis_binary` `int8`
- [x] **Business-rule validations pass** — all features non-negative, target ∈ {0,1}, 569 rows intact
- [x] **Cleaning log exists** — `_cleaning_log.csv`, one row per action with rows-affected and rationale (incl. zero-inflation flag & retained tails)
- [x] **Cleaned data saved to `data/processed/`** — `wdbc_clean.parquet` (569 × 33)

**→ Proceed to `03_eda.ipynb`.**